In [ ]:
import pandas as pd
import polars as pl
from loguru import logger
from pathlib import Path
import hydra
from functools import reduce
from omegaconf import DictConfig, OmegaConf

import hvplot.pandas
import holoviews as hv
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

tqdm.pandas()

from workspace_dani.data_preprocessing.utils.data_handling import load_parquet_table

hv.extension("bokeh")
#pd.options.plotting.backend = 'holoviews

project_root_path = Path.cwd().parents[2]  #/ Path("data_preprocessing") / Path("utils")



### Small Cohort (Cohort B)

In [ ]:
dataset_path = project_root_path / Path("intermediary_data") / Path("dataset") / Path("sepsis-type-b-cohort") / Path(
    "2024-05-31") / Path("build_dataset_ts.parquet")

diag_path = project_root_path / Path("intermediary_data/cohorted/sepsis-type-b/2024-05-30/diagnosis.parquet")
lab_path = project_root_path / Path("intermediary_data/cohorted/sepsis-type-b/2024-05-30/lab.parquet")
aper_path = project_root_path / Path("intermediary_data/cohorted/sepsis-type-b/2024-05-30/vital_aperiodic.parquet")
per_path = project_root_path / Path("intermediary_data/cohorted/sepsis-type-b/2024-05-30/vital_periodic.parquet")

In [ ]:
df_dataset_b = load_parquet_table(parquet_path=dataset_path, verbose=False)

df_diag_b = load_parquet_table(parquet_path=diag_path, verbose=False)
df_lab_b = load_parquet_table(parquet_path=lab_path, verbose=False)
df_aper_b = load_parquet_table(parquet_path=aper_path, verbose=False)
df_per_b = load_parquet_table(parquet_path=per_path, verbose=False)


In [ ]:
patient_ids_b = df_dataset_b["patientunitstayid"].unique().tolist()

Checking if every time stamp from each tables for each patient is found in the dataset


In [ ]:
sample_patient_id = patient_ids_b[0]

sample_dataset = df_dataset_b[df_dataset_b["patientunitstayid"] == sample_patient_id]

sample_diag = df_diag_b[df_diag_b["patientunitstayid"] == sample_patient_id]
sample_lab = df_lab_b[df_lab_b["patientunitstayid"] == sample_patient_id]
sample_per = df_per_b[df_per_b["patientunitstayid"] == sample_patient_id]
sample_aper = df_aper_b[df_aper_b["patientunitstayid"] == sample_patient_id]

sample_tables = [sample_diag, sample_lab, sample_aper, sample_per]

In [ ]:
sample_dataset_time_stamps: list = sample_dataset["time_offset"].unique().tolist()

In [ ]:
sample_tables_time_stamps_all = pd.concat(sample_tables)["time_offset"].unique().tolist()

In [ ]:
assert len(sample_tables_time_stamps_all) == len(sample_dataset_time_stamps), "Missing time_stamps in merge tables"

The function below runs pretty slow, not worth the effort to speed it up to actually useful speed, a single patient sample should be indicative of proper merging !


In [ ]:
def check_if_all_time_stamps_present_after_merge():
    
    def check_patient_timestamps(group):
        patient_id = group["patientunitstayid"].unique().tolist()[0]

        patient_dataset = df_dataset_b[df_dataset_b["patientunitstayid"] == patient_id]

        patient_diag = df_diag_b[df_diag_b["patientunitstayid"] == patient_id]
        patient_lab = df_lab_b[df_lab_b["patientunitstayid"] == patient_id]
        patient_per = df_per_b[df_per_b["patientunitstayid"] == patient_id]
        patient_aper = df_aper_b[df_aper_b["patientunitstayid"] == patient_id]

        patient_tables = [patient_diag, patient_lab, patient_per, patient_aper]

        patient_dataset_time_stamps: list = patient_dataset["time_offset"].unique().tolist()
        patient_tables_time_stamps_all = pd.concat(patient_tables)["time_offset"].unique().tolist()

        if len(patient_tables_time_stamps_all) != len(patient_dataset_time_stamps):
            error_message = f"Merge messed up for patient_id {patient_id}"
            # logger.error(error_message)
            print(error_message)
        else:
            # logger.info(f"Merged patient_id {patient_id} alright")
            print(f"Merged patient_id {patient_id} alright")

    df_dataset_b.groupby("patientunitstayid", group_keys=False).progress_apply(check_patient_timestamps)


#check_if_all_time_stamps_present_after_merge()


### Small Cohort (Cohort A)

In [ ]:
dataset_path = project_root_path / Path("intermediary_data") / Path("dataset") / Path("sepsis-type-a-cohort") / Path(
    "2024-05-31") / Path("build_dataset_ts.parquet")

diag_path = project_root_path / Path("intermediary_data/cohorted/sepsis-type-a/2024-05-31/diagnosis.parquet")
lab_path = project_root_path / Path("intermediary_data/cohorted/sepsis-type-a/2024-05-31/lab.parquet")
aper_path = project_root_path / Path("intermediary_data/cohorted/sepsis-type-a/2024-05-31/vital_aperiodic.parquet")
per_path = project_root_path / Path("intermediary_data/cohorted/sepsis-type-a/2024-05-31/vital_periodic.parquet")

In [ ]:


df_dataset_a = load_parquet_table(parquet_path=dataset_path, verbose=False)

df_diag_a = load_parquet_table(parquet_path=diag_path, verbose=False)
df_lab_a = load_parquet_table(parquet_path=lab_path, verbose=False)
df_aper_a = load_parquet_table(parquet_path=aper_path, verbose=False)
df_per_a = load_parquet_table(parquet_path=per_path, verbose=False)


In [ ]:
patient_ids_a = df_dataset_a["patientunitstayid"].unique().tolist()

In [ ]:
sample_patient_id_a = patient_ids_a[0]

sample_dataset_a = df_dataset_a[df_dataset_a["patientunitstayid"] == sample_patient_id_a]

sample_diag_a = df_diag_a[df_diag_a["patientunitstayid"] == sample_patient_id_a]
sample_lab_a = df_lab_a[df_lab_a["patientunitstayid"] == sample_patient_id_a]
sample_per_a = df_per_a[df_per_a["patientunitstayid"] == sample_patient_id_a]
sample_aper_a = df_aper_a[df_aper_a["patientunitstayid"] == sample_patient_id_a]

In [ ]:
sample_tables_a = [sample_diag_a, sample_lab_a, sample_aper_a, sample_per_a]
sample_dataset_time_stamps_a: list = sample_dataset_a["time_offset"].unique().tolist()
sample_tables_time_stamps_all_a = pd.concat(sample_tables_a)["time_offset"].unique().tolist()

In [ ]:
assert len(sample_tables_time_stamps_all_a) == len(sample_dataset_time_stamps_a), "Missing time_stamps in merge tables"


-> Tables merged with pandas LGTM